## Setup

### Imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ctgan import CTGAN
from ctgan import TVAE
from tabfairgan import TFG

import torch
import torch.nn as nn
import torch.optim as optim

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score
from sklearn.impute import SimpleImputer

c:\Users\fumme\School\fair-synthetic-data-gen\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Dataset

Sensitive attributes include:
- sex
- race

Target variable:
- two_year_recid

In [2]:
scores_two_years = pd.read_csv("data/compas-scores-two-years.csv")

columns_to_include = [
    "sex",
    "age",
    "age_cat",
    "race",
    "juv_fel_count",
    "juv_misd_count",
    "juv_other_count",
    "priors_count",
    "c_charge_degree",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
    "two_year_recid",
]

df = scores_two_years[columns_to_include]

n = df.shape[0]

main_features = ["sex", "race", "two_year_recid"]

In [3]:
categorical_columns = [
    "sex",
    "c_charge_degree",
    "two_year_recid",
    "age_cat",
    "race",
    "c_charge_desc",
    "r_charge_degree",
    "r_charge_desc",
]

In [4]:
def display_main_features_info(df: pd.DataFrame):
    about_main_feats = {}

    for column in main_features:
        temp = pd.DataFrame()

        temp["value"] = df[column].unique().tolist()
        temp["count"] = df[column].value_counts().tolist()
        temp["percentage"] = round(
            df[column].value_counts(normalize=True) * 100, 1
        ).tolist()
        about_main_feats[column] = temp
        print(f"\n\nfeature: {column}")
        print(about_main_feats[column])

In [8]:
from fairgan import FairGAN

fg = FairGAN(df, "sex", "two_year_recid", verbose=True)

fg.train()

Epoch 0 | D1: 0.7004 | D2: 15.5729 | G: 0.6978 | Fair: 0.6933
Epoch 10 | D1: 0.7376 | D2: 7.1925 | G: 0.6700 | Fair: 0.6935
Epoch 20 | D1: 0.6379 | D2: 12.0475 | G: 0.7923 | Fair: 0.6943
Epoch 30 | D1: 0.5384 | D2: 11.5242 | G: 0.9293 | Fair: 0.6956
Epoch 40 | D1: 0.5439 | D2: 8.6328 | G: 0.9182 | Fair: 0.6963
Epoch 50 | D1: 0.6089 | D2: 7.9575 | G: 0.8236 | Fair: 0.6970
Epoch 60 | D1: 0.6847 | D2: 6.3865 | G: 0.7404 | Fair: 0.6993
Epoch 70 | D1: 0.6842 | D2: 1.0133 | G: 0.7478 | Fair: 0.7020
Epoch 80 | D1: 0.5598 | D2: 0.6660 | G: 0.9173 | Fair: 0.7093
Epoch 90 | D1: 0.3521 | D2: 0.5366 | G: 1.3255 | Fair: 0.7095
Epoch 100 | D1: 0.2096 | D2: 0.5037 | G: 1.8067 | Fair: 0.6949
Epoch 110 | D1: 0.1726 | D2: 0.4821 | G: 2.0117 | Fair: 0.6949
Epoch 120 | D1: 0.1788 | D2: 0.4745 | G: 2.0008 | Fair: 0.6988
Epoch 130 | D1: 0.2353 | D2: 0.4678 | G: 1.7505 | Fair: 0.7027
Epoch 140 | D1: 0.3252 | D2: 0.4626 | G: 1.4983 | Fair: 0.7074
Epoch 150 | D1: 0.3353 | D2: 0.4578 | G: 1.4963 | Fair: 0.7118


## Data Preprocessing

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7214 entries, 0 to 7213
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   sex              7214 non-null   object
 1   age              7214 non-null   int64 
 2   age_cat          7214 non-null   object
 3   race             7214 non-null   object
 4   juv_fel_count    7214 non-null   int64 
 5   juv_misd_count   7214 non-null   int64 
 6   juv_other_count  7214 non-null   int64 
 7   priors_count     7214 non-null   int64 
 8   c_charge_degree  7214 non-null   object
 9   c_charge_desc    7185 non-null   object
 10  r_charge_degree  3471 non-null   object
 11  r_charge_desc    3413 non-null   object
 12  two_year_recid   7214 non-null   int64 
dtypes: int64(6), object(7)
memory usage: 732.8+ KB


In [ ]:
# TODO: Handle missing values in "r_charge_degree" and "r_charge_desc" (<50% missing)
# TODO: Handle missing values in "c_charge_desc" (>1% missing)

# Encode categorical variables
df_encoded = df.copy()

for col in df_encoded.columns:
    if df_encoded[col].dtype == "object":
        df_encoded[col] = LabelEncoder().fit_transform(df_encoded[col])

### FairGAN

#### Define Sensitive and Target

In [ ]:
S = "sex"
Y = "two_year_recid"

S_index = df_encoded.columns.get_loc(S)
Y_index = df_encoded.columns.get_loc(Y)

# Model Training

## CTGAN
The CTGAN model learns the joint distribution of the dataset.

In [ ]:
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "1"

In [ ]:
model = CTGAN(epochs=200, verbose=True)

model.fit(df, discrete_columns=categorical_columns)

## TVAE

TVAE learns a latent representation of the data and reconstructs it through a decoder network.

Unlike CTGAN, TVAE optimizes a probabilistic objective rather than using adversarial training.

In [ ]:
model = TVAE(epochs=200, verbose=True)

model.fit(df, discrete_columns=categorical_columns)

## FairGAN Model
### Model Architecture

FairGAN consists of:
- Generator
- Discriminator
- Fairness discriminator

In [ ]:
input_dim = df_encoded.shape[1]
noise_dim = 100


# Generator
class Generator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(noise_dim, 128), nn.ReLU(), nn.Linear(128, input_dim)
        )

    def forward(self, z):
        return self.model(z)


# Real/Fake Discriminator (D1)
class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)


# Fairness Discriminator (D2)
class FairnessDiscriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.model(x)


G = Generator()
D1 = Discriminator()
D2 = FairnessDiscriminator()

criterion = nn.BCELoss()

g_optimizer = optim.Adam(G.parameters(), lr=0.001)
d1_optimizer = optim.Adam(D1.parameters(), lr=0.001)
d2_optimizer = optim.Adam(D2.parameters(), lr=0.001)

data = torch.tensor(df_encoded.values, dtype=torch.float32)

epochs = 200
lambda_fair = 0.05

for epoch in range(epochs):

    # Train D1 (real vs fake)
    real_labels = torch.ones(len(data), 1)
    fake_labels = torch.zeros(len(data), 1)

    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    d1_real = D1(data)
    d1_fake = D1(fake_data.detach())

    d1_loss = criterion(d1_real, real_labels) + criterion(d1_fake, fake_labels)

    d1_optimizer.zero_grad()
    d1_loss.backward()
    d1_optimizer.step()

    # Train D2 (predict S)
    S_real = data[:, S_index].unsqueeze(1)

    d2_pred = D2(data)
    d2_loss = criterion(d2_pred, S_real)

    d2_optimizer.zero_grad()
    d2_loss.backward()
    d2_optimizer.step()

    # Train Generator
    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    # GAN loss (fool D1)
    g_loss = criterion(D1(fake_data), real_labels)

    # FAIRNESS loss (Xu-style)
    S_fake_pred = D2(fake_data)

    # push D2 to be uncertain (0.5)
    target_uncertain = torch.full_like(S_fake_pred, 0.5)
    fair_loss = criterion(S_fake_pred, target_uncertain)

    # TOTAL LOSS
    total_g_loss = g_loss - lambda_fair * fair_loss

    g_optimizer.zero_grad()
    total_g_loss.backward()
    g_optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} | D1: {d1_loss.item():.4f} | D2: {d2_loss.item():.4f} | G: {g_loss.item():.4f} | Fair: {fair_loss.item():.4f}"
        )

### Model Initialization

In [ ]:
G = Generator()
D1 = Discriminator()
D2 = FairnessDiscriminator()

criterion = nn.BCELoss()

g_optimizer = optim.Adam(G.parameters(), lr=0.001)
d1_optimizer = optim.Adam(D1.parameters(), lr=0.001)
d2_optimizer = optim.Adam(D2.parameters(), lr=0.001)

data = torch.tensor(df_encoded.values, dtype=torch.float32)

epochs = 200
lambda_fair = 0.05

for epoch in range(epochs):

    # Train D1 (real vs fake)
    real_labels = torch.ones(len(data), 1)
    fake_labels = torch.zeros(len(data), 1)

    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    d1_real = D1(data)
    d1_fake = D1(fake_data.detach())

    d1_loss = criterion(d1_real, real_labels) + criterion(d1_fake, fake_labels)

    d1_optimizer.zero_grad()
    d1_loss.backward()
    d1_optimizer.step()

    # Train D2 (predict S)
    S_real = data[:, S_index].unsqueeze(1)

    d2_pred = D2(data)
    d2_loss = criterion(d2_pred, S_real)

    d2_optimizer.zero_grad()
    d2_loss.backward()
    d2_optimizer.step()

    # Train Generator
    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    # GAN loss (fool D1)
    g_loss = criterion(D1(fake_data), real_labels)

    # FAIRNESS loss (Xu-style)
    S_fake_pred = D2(fake_data)

    # push D2 to be uncertain (0.5)
    target_uncertain = torch.full_like(S_fake_pred, 0.5)
    fair_loss = criterion(S_fake_pred, target_uncertain)

    # TOTAL LOSS
    total_g_loss = g_loss - lambda_fair * fair_loss

    g_optimizer.zero_grad()
    total_g_loss.backward()
    g_optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} | D1: {d1_loss.item():.4f} | D2: {d2_loss.item():.4f} | G: {g_loss.item():.4f} | Fair: {fair_loss.item():.4f}"
        )

## FairGAN Model Training

In [ ]:
for epoch in range(epochs):

    # Train D1 (real vs fake)
    real_labels = torch.ones(len(data), 1)
    fake_labels = torch.zeros(len(data), 1)

    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    d1_real = D1(data)
    d1_fake = D1(fake_data.detach())

    d1_loss = criterion(d1_real, real_labels) + criterion(d1_fake, fake_labels)

    d1_optimizer.zero_grad()
    d1_loss.backward()
    d1_optimizer.step()

    # Train D2 (predict S)
    S_real = data[:, S_index].unsqueeze(1)

    d2_pred = D2(data)
    d2_loss = criterion(d2_pred, S_real)

    d2_optimizer.zero_grad()
    d2_loss.backward()
    d2_optimizer.step()

    # Train Generator
    z = torch.randn(len(data), noise_dim)
    fake_data = G(z)

    # GAN loss (fool D1)
    g_loss = criterion(D1(fake_data), real_labels)

    # FAIRNESS loss (Xu-style)
    S_fake_pred = D2(fake_data)

    # push D2 to be uncertain (0.5)
    target_uncertain = torch.full_like(S_fake_pred, 0.5)
    fair_loss = criterion(S_fake_pred, target_uncertain)

    # TOTAL LOSS
    total_g_loss = g_loss - lambda_fair * fair_loss

    g_optimizer.zero_grad()
    total_g_loss.backward()
    g_optimizer.step()

    if epoch % 10 == 0:
        print(
            f"Epoch {epoch} | D1: {d1_loss.item():.4f} | D2: {d2_loss.item():.4f} | G: {g_loss.item():.4f} | Fair: {fair_loss.item():.4f}"
        )